# When do trips occur?

Uses the officially downloaded June 2024 dataset. No cycling paths are inferred here.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
DATA = PROJECT / 'data/processed/bikeshare_2024_06.parquet'
assert DATA.exists(), f'Missing {DATA}; run the download and prepare scripts from the repository root.'
df = pd.read_parquet(DATA)
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
print(f'Loaded {len(df):,} records')

In [ ]:
valid=df.dropna(subset=['Start_Time']).copy()
valid['day']=valid['Start_Time'].dt.day
valid['hour']=valid['Start_Time'].dt.hour
matrix=valid.groupby(['day','hour']).size().unstack(fill_value=0).reindex(index=range(1,31),columns=range(24),fill_value=0)
fig,ax=plt.subplots(figsize=(14,7))
im=ax.imshow(matrix.values,aspect='auto',origin='lower',cmap='magma',extent=[-.5,23.5,.5,30.5])
fig.colorbar(im,ax=ax,label='Recorded starts');ax.set(title='June 2024 | Trips by day and hour',xlabel='Hour',ylabel='Day of June');plt.show()

In [ ]:
valid['is_weekend']=valid['Start_Time'].dt.dayofweek>=5
hourly=valid.groupby(['is_weekend','hour']).size().unstack(0,fill_value=0)
days=valid.groupby('is_weekend')['Start_Time'].apply(lambda s:s.dt.date.nunique())
fig,ax=plt.subplots(figsize=(12,5))
for value,label in [(False,'Weekday'),(True,'Weekend')]:
    if value in hourly:
        ax.plot(hourly.index,hourly[value]/days[value],label=label)
ax.set(title='Average trips per day, by hour',xlabel='Hour',ylabel='Recorded trip starts per day');ax.legend();ax.grid(alpha=.2);plt.show()